# RAG System Demo
## Financial Document Question Answering

**System Mode:** CPU-first

This notebook demonstrates the complete RAG pipeline for answering questions from 10-K filings.

**Requirements:**
- Python 3.8+
- 8-10GB RAM
- Works on CPU

### 1. Setup and Installation

**Note:** This installs CPU-compatible versions by default.

In [1]:
import sys
sys.path.append('..')

import torch
from pathlib import Path
from loguru import logger

# Check system capabilities
print("="*60)
print("SYSTEM INFORMATION")
print("="*60)
print(f"GPU Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print("\n✅ GPU detected - you can use GPU mode for faster inference")
    print("   See CPU_SETUP.md for GPU configuration")
else:
    print("Device: CPU")
    print("\n✅ Running in CPU mode (default)")
    print("   Expected query time: 15-30 seconds")
    print("   RAM needed: 8-10GB")

print("="*60)



SYSTEM INFORMATION
GPU Available: False
Device: CPU

✅ Running in CPU mode (default)
   Expected query time: 15-30 seconds
   RAM needed: 8-10GB


### 2. Initialize RAG Pipeline

**Default Configuration:**
- Uses CPU (safe for all systems)
- Model: Phi-3-mini (3.8B params)
- FAISS: CPU version
- Memory: 8-10GB RAM

In [2]:
from src.config import RAGConfig
from src.main import initialize_pipeline, answer_question
from src.pipeline.rag_pipeline import RAGPipeline

print("Initializing RAG Pipeline with CPU-safe defaults...")
print("\nThis will:")
print("  1. Load PDF documents from data/pdfs/")
print("  2. Extract and chunk text")
print("  3. Generate embeddings (CPU)")
print("  4. Build FAISS index (CPU)")
print("  5. Load Phi-3-mini model")
print("\n⏱️  Expected time: 20-30 minutes on CPU\n")

# initialize_pipeline()

# Initialize pipeline
config = RAGConfig()
pipeline = RAGPipeline(config)

# Index documents
pdf_paths = list(Path("data/pdfs").glob("*.pdf"))
print(f"Found {len(pdf_paths)} PDF files:")
for pdf in pdf_paths:
    print(f"  - {pdf.name}")

print("\nIndexing documents...\n")
pipeline.index_documents(pdf_paths)

print("\n✅ Pipeline initialized and documents indexed successfully!")

W0218 02:31:16.452000 20176 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
2026-02-18 02:31:16.621 | INFO     | src.pipeline.rag_pipeline:__init__:37 - Initializing RAG Pipeline
2026-02-18 02:31:16.622 | INFO     | src.pipeline.rag_pipeline:_initialize_components:44 - Initializing pipeline components...
2026-02-18 02:31:16.622 | INFO     | src.services.embedding_service:__init__:28 - Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
2026-02-18 02:31:16.622 | INFO     | src.services.embedding_service:__init__:29 - Using device: cpu


Initializing RAG Pipeline with CPU-safe defaults...

This will:
  1. Load PDF documents from data/pdfs/
  2. Extract and chunk text
  3. Generate embeddings (CPU)
  4. Build FAISS index (CPU)
  5. Load Phi-3-mini model

⏱️  Expected time: 20-30 minutes on CPU



d:\01 Work\11-Side-Projects\02 RAG For Apple & Tesla\.venv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2026-02-18 02:31:19.186 | INFO     | src.services.embedding_service:__init__:39 - Embedding model loaded successfully
2026-02-18 02:31:19.188 | INFO     | src.services.vector_store:__init__:36 - Using FAISS CPU index
2026-02-18 02:31:19.188 | INFO     | src.services.vector_store:_initialize_index:42 - Initializing FAISS index with dimension 384
2026-02-18 02:31:19.190 | INFO     | src.services.retriever:_initialize_reranker:46 - Loading reranker model: cross-encoder/ms-marco-MiniLM-L-6-v2
2026-02-18 02:31:20.365 | INFO     | src.services.retriever:_initialize_reranker:52 - Reranker loaded successfully
2026-02-18 02:31:20.369 | INFO     | src.services.llm_service:__init__:

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

2026-02-18 02:31:29.364 | INFO     | src.services.llm_service:_load_model:136 - LLM loaded successfully
2026-02-18 02:31:29.364 | INFO     | src.pipeline.rag_pipeline:_initialize_components:70 - All components initialized successfully
2026-02-18 02:31:29.373 | INFO     | src.pipeline.rag_pipeline:index_documents:82 - Loading existing index...
2026-02-18 02:31:29.374 | INFO     | src.services.vector_store:load:182 - Loading vector store from data\vector_index
2026-02-18 02:31:29.387 | INFO     | src.services.vector_store:load:214 - Loaded 245 chunks from vector store
2026-02-18 02:31:29.389 | INFO     | src.pipeline.rag_pipeline:index_documents:85 - Loaded index with 245 chunks


Found 2 PDF files:
  - 10-Q4-2024-As-Filed.pdf
  - tsla-20231231-gen.pdf

Indexing documents...


✅ Pipeline initialized and documents indexed successfully!


In [3]:
# ALTERNATIVE: Explicitly use CPU-optimized config
# Uncomment to use this instead:

# from src.config_cpu import get_cpu_optimized_config
# config = get_cpu_optimized_config()
# print(f"Using config:")
# print(f"  Device: {config.llm.device}")
# print(f"  LLM: {config.llm.model_name}")
# print(f"  Embedding: {config.embedding.model_name}")
# print(f"  Batch size: {config.embedding.batch_size}")
# initialize_pipeline(config=config)

In [4]:
# ALTERNATIVE: GPU config (only if you have GPU)
# Requires: pip uninstall faiss-cpu && pip install faiss-gpu
# Uncomment to use GPU:

# if torch.cuda.is_available():
#     from src.config import RAGConfig
#     config = RAGConfig()
#     config.use_gpu = True
#     config.embedding.device = "cuda"
#     config.llm.device = "cuda"
#     config.llm.model_name = "mistralai/Mistral-7B-Instruct-v0.2"
#     config.llm.load_in_4bit = True
#     config.vector_store.use_gpu = True
#     initialize_pipeline(config=config)
# else:
#     print("⚠️ No GPU available. Using CPU config.")
#     initialize_pipeline()

### 3. Ask Questions

Now you can ask questions about Apple and Tesla 10-K filings.

**Expected response time:**
- CPU: 15-30 seconds per question
- GPU: 2-3 seconds per question

In [5]:
import time

question = "What was Apple's total revenue for fiscal year 2024?"
print(f"Question: {question}")
print("\n⏱️  Processing (this may take 15-30 seconds on CPU)...\n")

start_time = time.time()
result = pipeline.answer_question(question)
elapsed = time.time() - start_time

print("="*60)
print(f"Answer: {result['answer']}")
print(f"\nSources: {result['sources']}")
print(f"\nTime taken: {elapsed:.2f} seconds")
print("="*60)

2026-02-18 02:31:29.441 | INFO     | src.pipeline.rag_pipeline:answer_question:135 - Answering question: What was Apple's total revenue for fiscal year 2024?...
2026-02-18 02:31:29.441 | DEBUG    | src.services.retriever:retrieve:75 - Retrieving contexts for query: What was Apple's total revenue for fiscal year 2024?...
2026-02-18 02:31:29.442 | DEBUG    | src.services.embedding_service:encode:63 - Encoding 1 texts with batch size 32


Question: What was Apple's total revenue for fiscal year 2024?

⏱️  Processing (this may take 15-30 seconds on CPU)...



2026-02-18 02:31:30.517 | WARNING  | src.services.retriever:retrieve:91 - No results above similarity threshold


Answer: Not specified in the document.

Sources: []

Time taken: 1.08 seconds


In [6]:
# Example 2: Tesla question
question = "What types of vehicles does Tesla currently produce and deliver?"
print(f"Question: {question}\n")

start_time = time.time()
result = pipeline.answer_question(question)
elapsed = time.time() - start_time

print("="*60)
print(f"Answer: {result['answer']}")
print(f"\nSources: {result['sources']}")
print(f"\nTime taken: {elapsed:.2f} seconds")
print("="*60)

2026-02-18 02:31:30.542 | INFO     | src.pipeline.rag_pipeline:answer_question:135 - Answering question: What types of vehicles does Tesla currently produce and deliver?...
2026-02-18 02:31:30.543 | DEBUG    | src.services.retriever:retrieve:75 - Retrieving contexts for query: What types of vehicles does Tesla currently produce and deliver?...
2026-02-18 02:31:30.543 | DEBUG    | src.services.embedding_service:encode:63 - Encoding 1 texts with batch size 32
2026-02-18 02:31:30.560 | WARNING  | src.services.retriever:retrieve:91 - No results above similarity threshold


Question: What types of vehicles does Tesla currently produce and deliver?

Answer: Not specified in the document.

Sources: []

Time taken: 0.02 seconds


In [7]:
# Example 3: Out-of-scope question (should refuse)
question = "What is Tesla's stock price forecast for 2025?"
print(f"Question: {question}\n")

result = pipeline.answer_question(question)

print("="*60)
print(f"Answer: {result['answer']}")
print(f"\nSources: {result['sources']}")
print("\n✅ Should refuse with proper message (not in documents)")
print("="*60)

2026-02-18 02:31:30.577 | INFO     | src.pipeline.rag_pipeline:answer_question:135 - Answering question: What is Tesla's stock price forecast for 2025?...


Question: What is Tesla's stock price forecast for 2025?

Answer: This question cannot be answered based on the provided documents.

Sources: []

✅ Should refuse with proper message (not in documents)


### 4. Batch Processing

Process multiple questions efficiently.

In [9]:
test_questions = [
    {"question_id": 1, "question": "What was Apple's total revenue for fiscal year 2024?"},
    {"question_id": 6, "question": "What was Tesla's total revenue for 2023?"},
    {"question_id": 9, "question": "What types of vehicles does Tesla produce?"},
    {"question_id": 11, "question": "What is Tesla's stock price forecast for 2025?"}
]

print("Processing batch of questions...")
print(f"Total questions: {len(test_questions)}")
print(f"⏱️  Estimated time: {len(test_questions) * 20} - {len(test_questions) * 30} seconds on CPU\n")

start_time = time.time()
results = pipeline.batch_answer_questions(test_questions)
elapsed = time.time() - start_time

print("\n" + "="*60)
print("RESULTS")
print("="*60)
for r in results:
    print(f"\nQ{r['question_id']}: {r['answer'][:100]}...")
    print(f"Sources: {r['sources']}")

print(f"\n⏱️  Total time: {elapsed:.2f} seconds")
print(f"   Average: {elapsed/len(test_questions):.2f} seconds per question")
print("="*60)

2026-02-18 02:32:57.775 | INFO     | src.pipeline.rag_pipeline:batch_answer_questions:180 - Answering 4 questions in batch
2026-02-18 02:32:57.776 | INFO     | src.pipeline.rag_pipeline:batch_answer_questions:187 - Processing Q1: What was Apple's total revenue for fiscal year 2024?...
2026-02-18 02:32:57.777 | INFO     | src.pipeline.rag_pipeline:answer_question:135 - Answering question: What was Apple's total revenue for fiscal year 2024?...
2026-02-18 02:32:57.778 | DEBUG    | src.services.retriever:retrieve:75 - Retrieving contexts for query: What was Apple's total revenue for fiscal year 2024?...
2026-02-18 02:32:57.778 | DEBUG    | src.services.embedding_service:encode:63 - Encoding 1 texts with batch size 32
2026-02-18 02:32:57.797 | WARNING  | src.services.retriever:retrieve:91 - No results above similarity threshold
2026-02-18 02:32:57.799 | INFO     | src.pipeline.rag_pipeline:batch_answer_questions:187 - Processing Q6: What was Tesla's total revenue for 2023?...
2026-02-18 02

Processing batch of questions...
Total questions: 4
⏱️  Estimated time: 80 - 120 seconds on CPU


RESULTS

Q1: Not specified in the document....
Sources: []

Q6: Not specified in the document....
Sources: []

Q9: Not specified in the document....
Sources: []

Q11: This question cannot be answered based on the provided documents....
Sources: []

⏱️  Total time: 0.07 seconds
   Average: 0.02 seconds per question


### 5. Evaluation

Evaluate the quality of answers including hallucination detection.

In [10]:
from src.evaluation import RAGEvaluator

# Evaluate the batch results
evaluator = RAGEvaluator()
metrics = evaluator.evaluate(results)

# Print detailed report
evaluator.print_report(metrics)

2026-02-18 02:33:02.580 | WARNING  | src.evaluation:<module>:17 - rouge-score not available
2026-02-18 02:33:02.582 | INFO     | src.evaluation:evaluate:59 - Evaluating RAG results...
2026-02-18 02:33:02.582 | INFO     | src.evaluation:print_report:326 - ================================================================================
2026-02-18 02:33:02.583 | INFO     | src.evaluation:print_report:327 - EVALUATION REPORT
2026-02-18 02:33:02.583 | INFO     | src.evaluation:print_report:328 - ================================================================================
2026-02-18 02:33:02.585 | INFO     | src.evaluation:print_report:332 - 
Total Questions: 4
2026-02-18 02:33:02.586 | INFO     | src.evaluation:print_report:333 - In-Scope: 3, Out-of-Scope: 1
2026-02-18 02:33:02.586 | INFO     | src.evaluation:print_report:335 - 
--- In-Scope Performance ---
2026-02-18 02:33:02.587 | INFO     | src.evaluation:print_report:336 - Exact Match Rate: 0.00%
2026-02-18 02:33:02.587 | INFO     |

In [11]:
# View specific metrics
import json

print("\nAGGREGATE METRICS:")
print(json.dumps(metrics['aggregate'], indent=2))


AGGREGATE METRICS:
{
  "total_questions": 4,
  "in_scope_questions": 3,
  "out_of_scope_questions": 1,
  "exact_match_rate": 0.0,
  "key_info_accuracy": 0.0,
  "hallucination_rate": 0.0,
  "source_citation_rate": 0.0,
  "refusal_accuracy": 1.0,
  "overall_score": 0.5
}


### 6. Custom Questions

In [12]:
# Interactive question answering
def ask_interactive():
    """Interactive Q&A loop."""
    print("\n" + "="*60)
    print("INTERACTIVE MODE")
    print("="*60)
    print("Ask questions about Apple or Tesla 10-K filings.")
    print("Type 'quit' to exit.\n")
    
    while True:
        question = input("\nYour question: ").strip()
        
        if question.lower() in ['quit', 'exit', 'q']:
            print("\nGoodbye!")
            break
        
        if not question:
            continue
        
        print("\n⏱️  Processing...\n")
        start_time = time.time()
        result = answer_question(question)
        elapsed = time.time() - start_time
        
        print("="*60)
        print(f"Answer: {result['answer']}")
        print(f"\nSources: {result['sources']}")
        print(f"\nTime: {elapsed:.2f}s")
        print("="*60)

# Uncomment to run interactive mode
# ask_interactive()

In [14]:
# Or ask a single custom question
custom_question = "How many shares of common stock did Apple have outstanding?"

result = pipeline.answer_question(custom_question)

print(f"Question: {custom_question}")
print(f"\nAnswer: {result['answer']}")
print(f"\nSources: {result['sources']}")

2026-02-18 02:33:35.909 | INFO     | src.pipeline.rag_pipeline:answer_question:135 - Answering question: How many shares of common stock did Apple have outstanding?...
2026-02-18 02:33:35.910 | DEBUG    | src.services.retriever:retrieve:75 - Retrieving contexts for query: How many shares of common stock did Apple have outstanding?...
2026-02-18 02:33:35.911 | DEBUG    | src.services.embedding_service:encode:63 - Encoding 1 texts with batch size 32
2026-02-18 02:33:35.978 | WARNING  | src.services.retriever:retrieve:91 - No results above similarity threshold


Question: How many shares of common stock did Apple have outstanding?

Answer: Not specified in the document.

Sources: []


### 7. Pipeline Statistics

View information about the RAG pipeline.

In [16]:
# Get pipeline statistics
stats = pipeline.get_pipeline_stats()

print("="*60)
print("PIPELINE STATISTICS")
print("="*60)

print(f"\nIndexed: {stats['is_indexed']}")
print(f"\nConfiguration:")
for key, value in stats['config'].items():
    print(f"  {key}: {value}")

if 'vector_store' in stats:
    print(f"\nVector Store:")
    for key, value in stats['vector_store'].items():
        print(f"  {key}: {value}")

print("="*60)

PIPELINE STATISTICS

Indexed: True

Configuration:
  chunking_strategy: semantic
  chunk_size: 512
  embedding_model: sentence-transformers/all-MiniLM-L6-v2
  llm_model: microsoft/Phi-3-mini-4k-instruct
  use_reranker: True
  use_gpu: False

Vector Store:
  total_chunks: 245
  index_size: 245
  embedding_dim: 384
  use_gpu: False
  similarity_metric: cosine
